# UCI Real Estate Valuation
MDI3003 Advanced Predictive Analytics

**Student Name:** Harshita Bogineni

**Reg No:** 23MID0043

**Dataset:** UCI_Real_Estate

In [6]:
# Install once if needed
!pip install -q ucimlrepo joblib


In [21]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, platform, joblib
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split,KFold,cross_validate,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,PolynomialFeatures
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
SEED=42

In [16]:
import sklearn

np.random.seed(SEED)

print('Python:', platform.python_version())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)


Python: 3.11.0
pandas: 2.3.3
numpy: 1.26.4
scikit-learn: 1.8.0


## Load Dataset

In [8]:
from ucimlrepo import fetch_ucirepo


real_estate=fetch_ucirepo(id=477)
X=real_estate.data.features
y=real_estate.data.targets.iloc[:,0]
df=X.copy()
df['HousePrice']=y
display(df.head())
print(df.shape)


   X1 transaction date  X2 house age  X3 distance to the nearest MRT station  X4 number of convenience stores  X5 latitude  X6 longitude  HousePrice
0             2012.917          32.0                                84.87882                               10     24.98298     121.54024        37.9
1             2012.917          19.5                               306.59470                                9     24.98034     121.53951        42.2
2             2013.583          13.3                               561.98450                                5     24.98746     121.54391        47.3
3             2013.500          13.3                               561.98450                                5     24.98746     121.54391        54.8
4             2012.833           5.0                               390.56840                                5     24.97937     121.54245        43.1
(414, 7)


## Data Audit

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 7 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   X1 transaction date                     414 non-null    float64
 1   X2 house age                            414 non-null    float64
 2   X3 distance to the nearest MRT station  414 non-null    float64
 3   X4 number of convenience stores         414 non-null    int64  
 4   X5 latitude                             414 non-null    float64
 5   X6 longitude                            414 non-null    float64
 6   HousePrice                              414 non-null    float64
dtypes: float64(6), int64(1)
memory usage: 22.8 KB


In [10]:
df.describe().T

In [11]:
df.isnull().sum()

In [12]:
df.duplicated().sum()

In [13]:
df.dtypes

## Exploratory Data Analysis

In [14]:
sns.histplot(df['HousePrice'],kde=True);plt.show()
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True),annot=True,cmap='coolwarm')
plt.show()

for col in df.columns:
    plt.figure(figsize=(5,3))
    sns.boxplot(x=df[col]);plt.title(col);plt.show()

for col in X.columns:
    plt.figure(figsize=(5,3))
    sns.scatterplot(data=df,x=col,y='HousePrice')
    plt.show()

sns.pairplot(df)
plt.show()


## Preprocessing

In [17]:
num=X.columns
pre=ColumnTransformer([
('num',Pipeline([
('imp',SimpleImputer(strategy='median')),
('scale',StandardScaler())
]),num)
])

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


## Dummy Baseline

In [22]:
from sklearn.dummy import DummyRegressor
dummy=DummyRegressor()
dummy.fit(X_train,y_train)
p=dummy.predict(X_test)
mae=mean_absolute_error(y_test,p)
mse=mean_squared_error(y_test,p)
rmse=np.sqrt(mse)
r2=r2_score(y_test,p)
print('Baseline MAE:',mae)
print('Baseline MSE:',mse)
print('Baseline RMSE:',rmse)
print('Baseline R2:',r2)


Baseline MAE: 10.91674735194555
Baseline MSE: 171.96955286723147
Baseline RMSE: 13.113716211174903
Baseline R2: -0.025094270136263974


## Simple Linear Regression

In [23]:
slr=LinearRegression()
slr.fit(X_train[['X2 house age']],y_train)
pred=slr.predict(X_test[['X2 house age']])
print('R2',r2_score(y_test,pred))


R2 0.015227904910820378


## Multiple Regression Models

In [25]:
models={
'Linear Regression':LinearRegression(),
'Ridge':Ridge(),
'Lasso':Lasso(alpha=0.001,max_iter=20000),
'ElasticNet':ElasticNet(alpha=0.001,max_iter=20000),
'Decision Tree':DecisionTreeRegressor(random_state=42),
'Random Forest':RandomForestRegressor(n_estimators=300,random_state=42),
'Gradient Boosting':GradientBoostingRegressor(random_state=42)
}

results=[]
pipes={}
for n,m in models.items():
    pipe=Pipeline([('pre',pre),('model',m)])
    pipe.fit(X_train,y_train)
    pr=pipe.predict(X_test)
    mae=mean_absolute_error(y_test,pr)
    mse=mean_squared_error(y_test,pr)
    rmse=np.sqrt(mse)
    r2=r2_score(y_test,pr)
    Xt=pipe.named_steps['pre'].transform(X_test)
    n_samples,n_features=Xt.shape
    adj_r2=np.nan if n_samples<=n_features+1 else 1-(1-r2)*(n_samples-1)/(n_samples-n_features-1)
    results.append([n,mae,mse,rmse,r2,adj_r2])
    pipes[n]=pipe

results_df=pd.DataFrame(results,columns=['Model','MAE','MSE','RMSE','R2','Adjusted_R2']).sort_values('RMSE')
display(results_df)


               Model       MAE        MSE      RMSE        R2  Adjusted_R2
5      Random Forest  3.913142  32.136145  5.668875  0.808439     0.793316
6  Gradient Boosting  3.903628  34.155560  5.844276  0.796402     0.780328
1              Ridge  5.302237  53.448497  7.310848  0.681399     0.656246
3         ElasticNet  5.304791  53.493662  7.313936  0.681129     0.655955
2              Lasso  5.305206  53.500360  7.314394  0.681089     0.655912
0  Linear Regression  5.305356  53.505619  7.314754  0.681058     0.655878
4      Decision Tree  5.931325  66.471446  8.153002  0.603770     0.572489


## Polynomial Regression

In [26]:
poly=Pipeline([
('imp',SimpleImputer(strategy='median')),
('poly',PolynomialFeatures(degree=2,include_bias=False)),
('scale',StandardScaler()),
('model',Ridge())
])
feat=X.columns[:3]
poly.fit(X_train[feat],y_train)
print(poly.score(X_test[feat],y_test))


0.680174193488904


## Cross Validation

In [27]:
cv=KFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
for n,m in models.items():
    pipe=Pipeline([('pre',pre),('model',m)])
    sc=cross_validate(pipe,X_train,y_train,cv=cv,
                      scoring=['neg_root_mean_squared_error','r2'])
    rows.append([n,-sc['test_neg_root_mean_squared_error'].mean(),sc['test_r2'].mean()])
cv_df=pd.DataFrame(rows,columns=['Model','CV_RMSE','CV_R2'])
display(cv_df)


               Model    CV_RMSE     CV_R2
0  Linear Regression   9.110891  0.547792
1              Ridge   9.108281  0.548088
2              Lasso   9.110763  0.547805
3         ElasticNet   9.110474  0.547838
4      Decision Tree  10.738769  0.356090
5      Random Forest   7.830243  0.664277
6  Gradient Boosting   8.350247  0.619500


## Hyperparameter Tuning

In [28]:
grid=GridSearchCV(
Pipeline([('pre',pre),('model',Ridge())]),
{'model__alpha':[0.001,0.01,0.1,1,10,100]},
cv=5,
scoring='neg_root_mean_squared_error')
grid.fit(X_train,y_train)
print(grid.best_params_)
print(-grid.best_score_)


{'model__alpha': 10}
9.171916761199167


## Residual Analysis

In [29]:
best=pipes[results_df.iloc[0]['Model']]
pred=best.predict(X_test)
res=y_test-pred

plt.scatter(pred,res)
plt.axhline(0,color='red')
plt.show()

sns.histplot(res,kde=True)
plt.show()

plt.scatter(y_test,pred)
plt.plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],'r--')
plt.show()


## Feature Importance

In [30]:
m=best.named_steps['model']
if hasattr(m,'feature_importances_'):
    fi=pd.DataFrame({
        'Feature':best.named_steps['pre'].get_feature_names_out(),
        'Importance':m.feature_importances_
    }).sort_values('Importance',ascending=False)
    display(fi)
elif hasattr(m,'coef_'):
    coef=pd.DataFrame({
        'Feature':best.named_steps['pre'].get_feature_names_out(),
        'Coefficient':m.coef_
    })
    display(coef)


                                       Feature  Importance
2  num__X3 distance to the nearest MRT station    0.566968
1                            num__X2 house age    0.172670
4                             num__X5 latitude    0.118270
5                            num__X6 longitude    0.072300
0                     num__X1 transaction date    0.045376
3         num__X4 number of convenience stores    0.024416


## Save Outputs

In [31]:
results_df.to_csv('uci_model_comparison.csv',index=False)
cv_df.to_csv('uci_cross_validation.csv',index=False)
joblib.dump(best,'uci_real_estate_model.joblib')
